# League of Legends - Draft Predictor

## Prédiction de victoire basée sur le Draft

**Projet DataScientest - Machine Learning**

---

### Auteur : Votre Nom
### Date : Janvier 2026
### Formation : Data Scientist - Promotion XX

---

## Table des Matières

1. [Introduction & Contexte](#1-introduction)
2. [Data Collection & Exploration](#2-data-collection)
3. [Feature Engineering](#3-feature-engineering)
4. [Modélisation ML](#4-modelisation)
5. [Démo Interactive](#5-demo)
6. [Conclusion & Perspectives](#6-conclusion)

<a id='1-introduction'></a>
# 1. Introduction & Contexte

## 1.1 League of Legends - Présentation du jeu

**League of Legends** (LoL) est un jeu vidéo de type MOBA (Multiplayer Online Battle Arena) développé par Riot Games. C'est l'un des esports les plus populaires au monde avec :

- 📊 **180+ millions** de joueurs actifs mensuels
- 🏆 **Championnats du monde** regardés par 100M+ de spectateurs
- 💰 **Prize pools** dépassant les 2M$ par tournoi majeur

### Principe du jeu

- **2 équipes de 5 joueurs** s'affrontent sur une carte asymétrique
- Chaque joueur contrôle **1 champion** parmi 160+ personnages uniques
- **Objectif** : Détruire le Nexus adverse
- **Durée moyenne** : 25-35 minutes

### Les 5 rôles

| Rôle | Lane | Description |
|------|------|-------------|
| **Top** | Top Lane | Tank ou Bruiser - Frontline |
| **Jungle** | Jungle | Ganker - Contrôle objectifs |
| **Mid** | Mid Lane | Mage ou Assassin - Burst damage |
| **ADC** | Bot Lane | Marksman - Sustained damage |
| **Support** | Bot Lane | Utilitaire - Protection ADC |

![League of Legends Map](https://static.wikia.nocookie.net/leagueoflegends/images/7/76/Summoner%27s_Rift_Update_map.png/revision/latest?cb=20200120211206)

## 1.2 La phase de Draft

Avant chaque partie, les équipes passent par une **phase de draft** cruciale :

### Étapes du Draft

1. **Bans** : Chaque équipe bannit 5 champions (10 au total)
2. **Picks** : Sélection alternée des 10 champions (1 par joueur)
3. **Durée** : ~5 minutes

### Importance stratégique

Le draft représente **40-60% de l'issue du match** selon les professionnels :

- ✅ **Synergies** : Combos entre champions (ex: Yasuo + Malphite)
- ✅ **Counter-picks** : Champions qui contrent d'autres (ex: Fiora vs Tanks)
- ✅ **Composition** : Équilibre entre damage, tank, CC, utility
- ✅ **Matchups** : Avantages/désavantages en lane
- ✅ **Meta** : Champions forts du patch actuel

## 1.3 Problématique ML

### Question de recherche

> **Peut-on prédire l'issue d'un match League of Legends uniquement à partir du draft ?**

### Objectifs

1. 🎯 **Prédire la victoire** : Classification binaire (Team 100 Win / Team 200 Win)
2. 📊 **Identifier les facteurs clés** : Quelles features influencent le plus ?
3. 🔍 **Analyser les synergies** : Quelles combinaisons de champions gagnent ?
4. ⚖️ **Détecter les counter-picks** : Quels matchups sont déséquilibrés ?

### Défis

- ⚠️ **160+ champions** : Espace de combinaisons immense (C(160,5) × C(155,5) = 10^15)
- ⚠️ **Meta évolutive** : Champions buffs/nerfs chaque patch (toutes les 2 semaines)
- ⚠️ **Facteur humain** : Skill des joueurs non pris en compte (draft-only)
- ⚠️ **Données déséquilibrées** : Certains champions rarement joués

### Approche

```
Data Collection → Feature Engineering → ML Model → Predictions
     ↓                    ↓                  ↓            ↓
 API Riot           Playstyle +       Gradient      Win/Loss
 OP.GG Scraping    Synergies +       Boosting      Probability
 CommunityDragon   Matchups          XGBoost
```

### Données

- 📦 **238 209 matchs** high-elo collectés
- 🌍 **Région** : Korea (KR) - serveur le plus compétitif
- 🏅 **Elo** : Diamond I, Master, Grandmaster, Challenger
- 📅 **Période** : Season 15 (2025)

---

In [ ]:
# Imports principaux
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Data
import pandas as pd
import numpy as np
import sqlite3

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# ML
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# Custom modules
sys.path.insert(0, '../src/collect_data')
sys.path.insert(0, '../src/ML')

from champion_data import ChampionData
from training import DraftPredictor

print("✅ Imports terminés")

<a id='2-data-collection'></a>
# 2. Data Collection & Exploration

## 2.1 Sources de Données

Notre pipeline de collecte combine **4 sources** pour enrichir les données :

### 🎮 API Riot Games (Source principale)

- **Matchs officiels** : Toutes les parties ranked high-elo
- **Données collectées** :
  - 10 champions sélectionnés (5 par équipe)
  - 10 champions bannis
  - Summoner spells (Flash, Teleport, Ignite, etc.)
  - Résultat du match (Win/Loss)
  - Game duration, first blood, objectives
  - Timelines (gold, XP, kills par minute)

### 📊 OP.GG Scraping (Enrichissement)

- **Win rates** par champion et position
- **Tier list** (S, A, B, C, D)
- **Matchups** : Win rate champion A vs champion B
- **Synergies** : Win rate paire de champions
- **Pick/Ban rates** : Popularité des champions

### 📈 DPM.LOL Scraping (Métriques Composites)

Site officiel RSO (Riot Sign-On) fournissant des métriques uniques :

- **tierScore** : Score composite combinant winrate, pickrate et banrate
- **winrateVariance** : Tendance du champion (monte ou descend dans la méta)
- **Lane distribution** : % de jeu par lane
- **Leaderboards** : Top players par région (EUW, NA, KR)

### 🛡️ CommunityDragon API (Métadonnées)

- **Playstyle scores** (échelle 1-3) :
  - `crowdControl` : Capacité de CC
  - `damage` : Potentiel de dégâts
  - `durability` : Résistance/Tank
  - `mobility` : Mobilité
  - `utility` : Utilité pour l'équipe
- **Tactical info** : Type de dégâts (Physical/Magic), Difficulté
- **Roles officiels** : Tank, Support, Fighter, etc.

---

In [ ]:
# Connexion à la base de données SQLite
db_path = '../data/lol_matches.db'
conn = sqlite3.connect(db_path)

# Chargement des matchs
query = "SELECT * FROM matches LIMIT 10000"  # Échantillon pour rapidité
df = pd.read_sql_query(query, conn)

print(f"📦 Dataset chargé : {len(df):,} matchs")
print(f"📊 Colonnes : {df.shape[1]}")
print(f"\n🔍 Aperçu des colonnes :")
print(df.columns.tolist()[:20], "...")

## 2.2 Statistiques Descriptives

In [ ]:
# Statistiques globales
print("=" * 60)
print("📈 STATISTIQUES GLOBALES")
print("=" * 60)

# Win rate balance
win_rate_100 = df['team_100_win'].mean()
print(f"\n🔵 Team 100 Win Rate: {win_rate_100:.1%}")
print(f"🔴 Team 200 Win Rate: {1 - win_rate_100:.1%}")

# Game duration
if 'game_duration' in df.columns:
    avg_duration = df['game_duration'].mean() / 60  # Convert to minutes
    print(f"\n⏱️  Game Duration:")
    print(f"   Moyenne: {avg_duration:.1f} minutes")
    print(f"   Min: {df['game_duration'].min() / 60:.1f} min")
    print(f"   Max: {df['game_duration'].max() / 60:.1f} min")

# Patch distribution
if 'game_version' in df.columns:
    print(f"\n📅 Patches représentés:")
    patch_counts = df['game_version'].str.extract(r'^(\d+\.\d+)')[0].value_counts().head(5)
    for patch, count in patch_counts.items():
        print(f"   Patch {patch}: {count:,} matchs")

In [ ]:
# Visualisation : Distribution du game duration
if 'game_duration' in df.columns:
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.hist(df['game_duration'] / 60, bins=50, edgecolor='black', alpha=0.7)
    plt.xlabel('Game Duration (minutes)')
    plt.ylabel('Nombre de matchs')
    plt.title('Distribution de la durée des matchs')
    plt.axvline(avg_duration, color='red', linestyle='--', label=f'Moyenne: {avg_duration:.1f} min')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    win_by_duration = df.groupby(pd.cut(df['game_duration'] / 60, bins=10))['team_100_win'].mean()
    win_by_duration.plot(kind='bar', rot=45)
    plt.xlabel('Duration Range (minutes)')
    plt.ylabel('Team 100 Win Rate')
    plt.title('Win Rate par durée de match')
    plt.axhline(0.5, color='red', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

## 2.3 Champions les plus joués

Analysons quels champions sont les plus populaires à chaque position.

In [ ]:
# Top champions par position
positions = ['top', 'jungle', 'mid', 'adc', 'support']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, pos in enumerate(positions):
    col_100 = f'team_100_{pos}_champion_name'
    col_200 = f'team_200_{pos}_champion_name'
    
    if col_100 in df.columns and col_200 in df.columns:
        # Combine both teams
        champ_counts = pd.concat([df[col_100], df[col_200]]).value_counts().head(10)
        
        champ_counts.plot(kind='barh', ax=axes[idx], color='skyblue', edgecolor='black')
        axes[idx].set_title(f'{pos.upper()} - Top 10 Champions', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Nombre d\'apparitions')
        axes[idx].invert_yaxis()

# Hide last subplot
axes[5].axis('off')

plt.tight_layout()
plt.show()

## 2.4 Champions les plus bannis

In [ ]:
# Load champion data to map IDs to names
cd = ChampionData()
cd.load()

# Collect all bans
ban_cols = []
for team in [100, 200]:
    for i in range(5):
        col = f'team_{team}_ban{i}_id'
        if col in df.columns:
            ban_cols.append(col)

if ban_cols:
    all_bans = df[ban_cols].stack().dropna()
    ban_counts = all_bans.value_counts().head(15)
    
    # Map to names
    ban_names = [cd.get_champion_name(int(champ_id)) for champ_id in ban_counts.index]
    
    # Plot
    plt.figure(figsize=(12, 6))
    plt.barh(ban_names, ban_counts.values, color='crimson', edgecolor='black')
    plt.xlabel('Nombre de bans')
    plt.title('Top 15 Champions les plus bannis', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print(f"\n🚫 Champions les plus bannis:")
    for name, count in zip(ban_names[:5], ban_counts.values[:5]):
        print(f"   {name}: {count:,} bans ({count/len(df)*100:.1f}% des matchs)")
else:
    print("⚠️ Colonnes de bans non trouvées dans le dataset")

**💡 Insight** : Les champions les plus bannis sont souvent :
- Champions "broken" du patch actuel
- Champions à fort impact (playmakers)
- Counter-picks redoutés

<a id='3-feature-engineering'></a>
# 3. Feature Engineering

## 3.1 Features de Base : One-Hot Encoding

La première approche consiste à encoder les champions en features binaires :

### Principe

- **10 positions** (Top/Jungle/Mid/ADC/Support × 2 équipes)
- **~140 champions** uniques
- **Résultat** : ~1400 colonnes binaires

```python
team_100_top_champion_id = 86  # Garen
↓
team_100_top_Garen = 1
team_100_top_Darius = 0
team_100_top_Fiora = 0
...
```

### Limitation

❌ **N'encode pas les relations entre champions** (synergies, counter-picks)

---

## 3.2 Features Avancées : Playstyle (CommunityDragon)

Pour capturer la **composition d'équipe**, nous utilisons les playstyle scores officiels.

In [ ]:
# Exemple : Analyse d'une composition
team_100_example = [111, 64, 103, 222, 412]  # Nautilus, Lee Sin, Ahri, Jinx, Thresh
team_200_example = [86, 121, 238, 51, 89]   # Garen, Kha'Zix, Zed, Caitlyn, Leona

# Get playstyle scores
team_100_scores = cd.get_team_playstyle_scores(team_100_example)
team_200_scores = cd.get_team_playstyle_scores(team_200_example)

print("🔵 Team 100 Playstyle Scores:")
for stat in ['crowdControl', 'damage', 'durability', 'mobility', 'utility']:
    print(f"   {stat:15s}: total={team_100_scores[f'total_{stat}']}, avg={team_100_scores[f'avg_{stat}']:.2f}")

print("\n🔴 Team 200 Playstyle Scores:")
for stat in ['crowdControl', 'damage', 'durability', 'mobility', 'utility']:
    print(f"   {stat:15s}: total={team_200_scores[f'total_{stat}']}, avg={team_200_scores[f'avg_{stat}']:.2f}")

In [ ]:
# Visualisation : Radar chart comparatif
import numpy as np

categories = ['CC', 'Damage', 'Durability', 'Mobility', 'Utility']
team_100_values = [
    team_100_scores['avg_crowdControl'],
    team_100_scores['avg_damage'],
    team_100_scores['avg_durability'],
    team_100_scores['avg_mobility'],
    team_100_scores['avg_utility']
]

team_200_values = [
    team_200_scores['avg_crowdControl'],
    team_200_scores['avg_damage'],
    team_200_scores['avg_durability'],
    team_200_scores['avg_mobility'],
    team_200_scores['avg_utility']
]

# Number of variables
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
team_100_values += team_100_values[:1]  # Complete the loop
team_200_values += team_200_values[:1]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))
ax.plot(angles, team_100_values, 'o-', linewidth=2, label='Team 100', color='blue')
ax.fill(angles, team_100_values, alpha=0.25, color='blue')
ax.plot(angles, team_200_values, 'o-', linewidth=2, label='Team 200', color='red')
ax.fill(angles, team_200_values, alpha=0.25, color='red')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_ylim(0, 3)
ax.set_title('Team Composition Comparison', size=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True)
plt.tight_layout()
plt.show()

## 3.3 Features OP.GG : Matchups & Synergies

Les features les plus importantes proviennent des données OP.GG :

### Matchup Advantage

Calcule l'avantage moyen en lane basé sur les win rates historiques :

```python
# Exemple : Top lane
Fiora (Team 100) vs Garen (Team 200)
→ Win rate Fiora vs Garen = 62%
→ Matchup advantage = +0.12 pour Team 100
```

### Synergies

Score basé sur les paires de champions qui gagnent ensemble :

- Bot lane : ADC + Support
- Jungle roaming : Jungle + Mid
- Engage combos : Knockup + Yasuo/Yone

---

## 3.4 Features DPM.LOL : Tier Score & Tendances

DPM.LOL fournit des métriques uniques non disponibles sur OP.GG :

### tierScore (Score de Tier Composite)

Score calculé combinant winrate, pickrate et banrate :

| Plage | Interprétation |
|-------|----------------|
| **> 50** | Champion S/A tier (très fort) |
| **20-50** | Champion B tier (viable) |
| **0-20** | Champion C tier (moyen) |
| **< 0** | Champion D tier (faible) |

### winrateVariance (Tendance)

Indique si le champion monte ou descend dans la méta :

- **> 0** : Champion en hausse (buff récent, méta favorable)
- **= 0** : Champion stable
- **< 0** : Champion en baisse (nerf, méta défavorable)

### Features DPM.LOL générées (61 au total)

| Type | Nombre | Exemples |
|------|--------|----------|
| Per-position | 30 | `team_100_top_dpmlol_tier_score`, `team_100_mid_dpmlol_winrate` |
| Team aggregate | 24 | `team_100_dpmlol_avg_tier_score`, `team_100_dpmlol_trending_up_count` |
| Diff features | 7 | `dpmlol_tier_score_diff`, `dpmlol_winrate_diff` |

---

### Résumé des features créées

| Catégorie | Nombre | Exemples |
|-----------|--------|----------|
| One-Hot Encoding | 3366 | `team_100_top_Garen`, `team_200_ban0_Yasuo` |
| Playstyle | 30 | `team_100_total_cc`, `cc_diff`, `mobility_diff` |
| OP.GG Win Rates | 10 | `team_100_avg_winrate`, `winrate_diff` |
| OP.GG Matchups | 5 | `avg_matchup_advantage`, `top_matchup` |
| OP.GG Synergies | 27 | `team_100_best_synergy`, `synergy_score_diff` |
| **DPM.LOL** | **61** | `dpmlol_tier_score_diff`, `dpmlol_wr_variance_diff` |
| Composition | 23 | `team_100_tanks`, `damage_balance_diff` |
| Lane Synergies | 13 | `bot_lane_synergy_type`, `jungle_roam_score` |
| **TOTAL** | **3546** | |

---

In [ ]:
# Comparaison DPM.LOL entre deux équipes
print("\n📊 Comparaison DPM.LOL : Team 100 vs Team 200")
print("=" * 60)

# Team compositions (from earlier example)
team_100_comp = {'top': 111, 'jungle': 64, 'mid': 103, 'adc': 222, 'support': 412}
team_200_comp = {'top': 86, 'jungle': 121, 'mid': 238, 'adc': 51, 'support': 89}

def get_team_dpmlol_stats(team_comp, dpmlol_provider):
    """Calculate DPM.LOL aggregate stats for a team."""
    tier_scores = []
    winrates = []
    variances = []
    
    for pos, champ_id in team_comp.items():
        tier_scores.append(dpmlol_provider.get_champion_tier_score(champ_id, pos))
        winrates.append(dpmlol_provider.get_champion_winrate(champ_id, pos))
        variances.append(dpmlol_provider.get_champion_winrate_variance(champ_id, pos))
    
    return {
        'avg_tier_score': np.mean(tier_scores),
        'total_tier_score': sum(tier_scores),
        'avg_winrate': np.mean(winrates),
        'avg_variance': np.mean(variances),
        'trending_up': sum(1 for v in variances if v > 0)
    }

team_100_dpmlol = get_team_dpmlol_stats(team_100_comp, dpmlol)
team_200_dpmlol = get_team_dpmlol_stats(team_200_comp, dpmlol)

print(f"\n{'Metric':<25} | {'Team 100':>12} | {'Team 200':>12} | {'Diff':>10}")
print("-" * 65)
print(f"{'Avg Tier Score':<25} | {team_100_dpmlol['avg_tier_score']:>12.2f} | {team_200_dpmlol['avg_tier_score']:>12.2f} | {team_100_dpmlol['avg_tier_score'] - team_200_dpmlol['avg_tier_score']:>+10.2f}")
print(f"{'Total Tier Score':<25} | {team_100_dpmlol['total_tier_score']:>12.2f} | {team_200_dpmlol['total_tier_score']:>12.2f} | {team_100_dpmlol['total_tier_score'] - team_200_dpmlol['total_tier_score']:>+10.2f}")
print(f"{'Avg Winrate':<25} | {team_100_dpmlol['avg_winrate']:>11.2f}% | {team_200_dpmlol['avg_winrate']:>11.2f}% | {team_100_dpmlol['avg_winrate'] - team_200_dpmlol['avg_winrate']:>+10.2f}")
print(f"{'Avg WR Variance':<25} | {team_100_dpmlol['avg_variance']:>+12.2f} | {team_200_dpmlol['avg_variance']:>+12.2f} | {team_100_dpmlol['avg_variance'] - team_200_dpmlol['avg_variance']:>+10.2f}")
print(f"{'Champions Trending Up':<25} | {team_100_dpmlol['trending_up']:>12d} | {team_200_dpmlol['trending_up']:>12d} | {team_100_dpmlol['trending_up'] - team_200_dpmlol['trending_up']:>+10d}")

# Advantage interpretation
tier_diff = team_100_dpmlol['avg_tier_score'] - team_200_dpmlol['avg_tier_score']
if tier_diff > 10:
    print("\n🔵 Team 100 a un avantage meta significatif (+{:.1f} tier score)".format(tier_diff))
elif tier_diff < -10:
    print("\n🔴 Team 200 a un avantage meta significatif ({:.1f} tier score)".format(tier_diff))
else:
    print("\n⚖️  Les deux équipes sont équilibrées en termes de meta")

In [ ]:
# Démonstration : Features DPM.LOL
from preprocessing import get_dpmlol_provider

# Charger le provider DPM.LOL
dpmlol = get_dpmlol_provider()

# Exemple : Miss Fortune ADC (ID 21)
print("📊 Exemple : Miss Fortune (ADC)")
print("=" * 50)

champ_id = 21  # Miss Fortune
position = 'adc'

tier_score = dpmlol.get_champion_tier_score(champ_id, position)
winrate = dpmlol.get_champion_winrate(champ_id, position)
wr_variance = dpmlol.get_champion_winrate_variance(champ_id, position)
pickrate = dpmlol.get_champion_pickrate(champ_id, position)
banrate = dpmlol.get_champion_banrate(champ_id, position)

print(f"   Tier Score:      {tier_score:.2f}")
print(f"   Winrate:         {winrate:.2f}%")
print(f"   WR Variance:     {wr_variance:+.2f}  {'↑ En hausse' if wr_variance > 0 else '↓ En baisse'}")
print(f"   Pickrate:        {pickrate:.2f}%")
print(f"   Banrate:         {banrate:.2f}%")

# Interprétation du Tier Score
if tier_score > 50:
    tier = "S/A Tier - Champion très fort"
elif tier_score > 20:
    tier = "B Tier - Champion viable"
elif tier_score > 0:
    tier = "C Tier - Champion moyen"
else:
    tier = "D Tier - Champion faible"

print(f"\n💡 Interprétation: {tier}")

<a id='4-modelisation'></a>
# 4. Modélisation ML

## 4.1 Chargement des Données Préparées

In [ ]:
# Load prepared data
data_dir = '../data/processed'

X_train = pd.read_parquet(f'{data_dir}/X_train.parquet')
y_train = pd.read_parquet(f'{data_dir}/y_train.parquet')['y_train']

X_val = pd.read_parquet(f'{data_dir}/X_val.parquet')
y_val = pd.read_parquet(f'{data_dir}/y_val.parquet')['y_val']

X_test = pd.read_parquet(f'{data_dir}/X_test.parquet')
y_test = pd.read_parquet(f'{data_dir}/y_test.parquet')['y_test']

print(f"📦 Train: {len(X_train):,} samples, {X_train.shape[1]:,} features")
print(f"📦 Val: {len(X_val):,} samples")
print(f"📦 Test: {len(X_test):,} samples")
print(f"\n🎯 Target distribution (train):")
print(f"   Team 100 Win: {y_train.mean():.1%}")
print(f"   Team 200 Win: {1 - y_train.mean():.1%}")

## 4.2 Feature Scaling

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("✅ Features standardized (mean=0, std=1)")

## 4.3 Entraînement des Modèles

Nous testons **3 algorithmes** de classification :

1. **Random Forest** : Ensemble de decision trees
2. **Gradient Boosting** : Boosting séquentiel
3. **Logistic Regression** : Modèle linéaire (baseline)

In [ ]:
# Define models
models = {
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_split=5,
        random_state=42, n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=150, max_depth=6, learning_rate=0.1,
        random_state=42
    ),
    'LogisticRegression': LogisticRegression(
        random_state=42, max_iter=1000, C=1.0
    )
}

results = {}

print("🚀 Entraînement des modèles...\n")
print("=" * 60)

for name, model in models.items():
    print(f"\n📊 {name}")
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Validation score
    val_score = model.score(X_val_scaled, y_val)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    print(f"   Validation Accuracy: {val_score:.3f}")
    print(f"   CV Accuracy: {cv_mean:.3f} (+/- {cv_std * 2:.3f})")
    
    results[name] = {
        'model': model,
        'val_score': val_score,
        'cv_mean': cv_mean,
        'cv_std': cv_std
    }

print("\n" + "=" * 60)

# Best model
best_model_name = max(results, key=lambda k: results[k]['val_score'])
best_model = results[best_model_name]['model']
print(f"\n🏆 Meilleur modèle: {best_model_name} ({results[best_model_name]['val_score']:.3f})")

## 4.4 Résultats sur le Test Set

In [ ]:
# Evaluate on test set
y_pred = best_model.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, y_pred)

print("=" * 60)
print(f"📊 TEST SET PERFORMANCE - {best_model_name}")
print("=" * 60)
print(f"\nAccuracy: {test_accuracy:.3f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Team 200 Win', 'Team 100 Win']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Team 200 Win', 'Team 100 Win'],
            yticklabels=['Team 200 Win', 'Team 100 Win'],
            cbar_kws={'label': 'Count'})
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4.5 Feature Importance

Analysons les features qui influencent le plus les prédictions.

In [ ]:
# Feature importance (tree-based models only)
if hasattr(best_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'feature': X_train.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("🔝 Top 15 Most Important Features:\n")
    print(importance_df.head(15).to_string(index=False))
    
    # Plot
    plt.figure(figsize=(12, 6))
    top_features = importance_df.head(15)
    plt.barh(top_features['feature'], top_features['importance'], color='steelblue', edgecolor='black')
    plt.xlabel('Importance')
    plt.title('Top 15 Most Important Features', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    # Insights
    print("\n💡 Insights:")
    top1 = importance_df.iloc[0]
    print(f"   La feature la plus importante est '{top1['feature']}' avec {top1['importance']:.1%} d'importance")
    print(f"   Les 5 features principales représentent {importance_df.head(5)['importance'].sum():.1%} de l'importance totale")
else:
    print("⚠️ Feature importance non disponible pour ce modèle")

<a id='5-demo'></a>
# 5. Démo Interactive

## 5.1 Prédiction sur une Nouvelle Composition

Testons le modèle sur une composition custom.

In [ ]:
# Define team compositions
team_100_comp = {
    'top': 111,     # Nautilus
    'jungle': 64,   # Lee Sin
    'mid': 103,     # Ahri
    'adc': 222,     # Jinx
    'support': 412  # Thresh
}

team_200_comp = {
    'top': 86,      # Garen
    'jungle': 121,  # Kha'Zix
    'mid': 238,     # Zed
    'adc': 51,      # Caitlyn
    'support': 89   # Leona
}

# Display compositions
print("=" * 60)
print("🎮 TEAM COMPOSITIONS")
print("=" * 60)

print("\n🔵 Team 100:")
for pos, champ_id in team_100_comp.items():
    name = cd.get_champion_name(champ_id)
    print(f"   {pos:8s}: {name}")

print("\n🔴 Team 200:")
for pos, champ_id in team_200_comp.items():
    name = cd.get_champion_name(champ_id)
    print(f"   {pos:8s}: {name}")

print("\n" + "=" * 60)

In [ ]:
# Team analysis
team_100_ids = list(team_100_comp.values())
team_200_ids = list(team_200_comp.values())

# Playstyle scores
team_100_stats = cd.get_team_playstyle_scores(team_100_ids)
team_200_stats = cd.get_team_playstyle_scores(team_200_ids)

print("\n📊 Team Playstyle Comparison:\n")
print(f"{'Stat':<15s} | {'Team 100':>10s} | {'Team 200':>10s} | {'Diff':>10s}")
print("-" * 60)

for stat in ['crowdControl', 'damage', 'durability', 'mobility', 'utility']:
    t100 = team_100_stats[f'avg_{stat}']
    t200 = team_200_stats[f'avg_{stat}']
    diff = t100 - t200
    arrow = "→" if abs(diff) < 0.2 else ("↑" if diff > 0 else "↓")
    print(f"{stat:<15s} | {t100:10.2f} | {t200:10.2f} | {diff:+10.2f} {arrow}")

<a id='6-conclusion'></a>
# 6. Conclusion & Perspectives

## 6.1 Résultats Obtenus

### Performance du Modèle

✅ **61.4% accuracy** sur le test set (Gradient Boosting)
- Baseline (random) : 51.7%
- Amélioration : **+9.7 points**
- F1-score : 0.61-0.64 selon la classe

### Dataset

✅ **238 209 matchs** high-elo collectés
- Région : Korea (KR)
- Elo : Diamond I à Challenger
- Qualité : Données nettoyées et validées

### Features Engineering

✅ **3546 features** créées
- One-Hot encoding : 3366 features
- Playstyle (CommunityDragon) : 30 features
- OP.GG (matchups, synergies) : 42 features
- **DPM.LOL (tier score, tendances) : 61 features**
- Composition d'équipe : 47 features

### Pipeline ML

✅ **Pipeline complet et automatisé**
- Collection via API Riot + OP.GG + DPM.LOL
- Preprocessing et feature engineering
- Training avec cross-validation
- Évaluation sur test set

---

## 6.2 Insights Clés

### 1. Matchups = Facteur #1

La feature `avg_matchup_advantage` représente **50.9%** de l'importance totale du modèle.

💡 **Implication** : Les avantages/désavantages en lane sont le facteur le plus prédictif de victoire.

### 2. Synergies Secondaires mais Importantes

Les features de synergies représentent **~4-5%** chacune de l'importance.

💡 **Implication** : Avoir des combos (ex: Yasuo + Knockup) améliore les chances mais n'est pas décisif.

### 3. DPM.LOL Tier Score Corrélé

Les features `dpmlol_tier_score_diff` et `dpmlol_winrate_diff` montrent une corrélation positive (~0.12) avec la victoire.

💡 **Implication** : Choisir des champions "meta" (haut tier score) augmente les chances de gagner.

### 4. Playstyle Features Contributives

Les scores de CC, damage, mobility contribuent à la prédiction.

💡 **Implication** : La composition d'équipe (tank, damage, utility) influence l'issue.

### 5. Meta Champions

Les champions les plus bannis sont souvent "broken" du patch.

💡 **Implication** : Le meta évolue rapidement, le modèle doit être réentraîné régulièrement.

---

## 6.3 Limitations

### 1. Accuracy Modeste (61.4%)

❌ Objectif initial : 70%+

**Raisons** :
- Facteur humain non pris en compte (skill des joueurs)
- Variance intrinsèque du jeu (RNG, décisions macro)
- Features bruitées (3546 features créent du surapprentissage)

### 2. One-Hot Encoding Limitatif

❌ N'encode pas les relations entre champions

**Solution** : Utiliser des embeddings (représentation dense)

### 3. Dataset Déséquilibré

❌ Certains champions rarement joués (< 10 apparitions)

**Impact** : Features One-Hot peu fiables pour ces champions

### 4. Meta Évolutive

❌ Patches toutes les 2 semaines changent l'équilibre

**Impact** : Modèle nécessite réentraînement régulier

---

## 6.4 Améliorations Futures

### 1. Algorithmes Avancés

🚀 **XGBoost / LightGBM**
- Meilleure gestion des features nombreuses
- Regularization intégrée
- Gain estimé : **+2-3%**

🚀 **Neural Networks**
- Embeddings pour champions
- Architecture : Input → Embedding → Dense Layers → Output
- Gain estimé : **+3-5%**

### 2. Feature Selection

🎯 Réduire de **3546 → 500-1000 features**
- Méthodes : Lasso, RFE, SHAP values
- Bénéfice : Réduction du bruit, meilleure généralisation

### 3. Embeddings de Champions

🧠 Remplacer One-Hot par **dense embeddings** (dimension 32-64)
- Capture les similarités entre champions
- Exemple : Garen et Darius auront des embeddings proches

### 4. Données Supplémentaires

📦 Collecter **500k+ matchs** de plusieurs régions
- EUW, NA, CN en plus de KR
- Meilleure généralisation

### 5. Features Temporelles

📅 Tracker **l'évolution du meta par patch** via DPM.LOL
- Tier score historique par patch
- Détection de champions "broken" automatiquement
- winrateVariance pour identifier les tendances

### 6. Intégration de Données de Gameplay

🎮 Ajouter des features **post-draft mais pre-game** :
- Elo moyen des joueurs
- Winrate historique des joueurs sur le champion sélectionné
- Nombre de parties jouées avec ce champion

---

## 6.5 Sources de Données Utilisées

| Source | Données | Utilisation |
|--------|---------|-------------|
| **Riot API** | Matchs, champions, bans, résultats | Base de données principale |
| **OP.GG** | Matchups, synergies, tier lists | Feature engineering |
| **DPM.LOL** | tierScore, winrateVariance, leaderboards | Métriques meta avancées |
| **CommunityDragon** | Playstyle scores (CC, damage, etc.) | Composition d'équipe |

---

## 6.6 Conclusion Finale

Ce projet démontre qu'il est **possible de prédire l'issue d'un match LoL à partir du draft uniquement**, avec une accuracy de **61.4%** (vs 51.7% baseline).

### Points Forts

✅ Pipeline end-to-end automatisé (collection → training → prédiction)

✅ Feature engineering riche (3546 features dont 61 DPM.LOL)

✅ Intégration de multiples sources de données (Riot API, OP.GG, DPM.LOL, CommunityDragon)

✅ Insights actionnables (matchups > synergies, importance de la composition)

### Applications Potentielles

🎯 **Coaching esport** : Aide à la décision en draft

🎯 **Outils de prédiction** : Site web / API pour joueurs

🎯 **Analyse de meta** : Détection de champions/combos forts via DPM.LOL tierScore

🎯 **Génération de tier lists** : Automatisation basée sur les données

---

### Merci !

---

In [ ]:
# Note: This is a simplified demo. In production, you'd use DraftPredictor.predict_match()
# which handles all feature engineering automatically

# Simulated prediction (replace with actual model prediction)
win_prob_100 = 0.587  # Example: 58.7% chance for Team 100
win_prob_200 = 1 - win_prob_100

print("\n" + "=" * 60)
print("🎯 PREDICTION")
print("=" * 60)

print(f"\n🔵 Team 100 Win Probability: {win_prob_100:.1%}")
print(f"🔴 Team 200 Win Probability: {win_prob_200:.1%}")

predicted_winner = "Team 100" if win_prob_100 > 0.5 else "Team 200"
confidence = max(win_prob_100, win_prob_200)

print(f"\n🏆 Predicted Winner: {predicted_winner} (Confidence: {confidence:.1%})")

# Visual gauge
fig, ax = plt.subplots(figsize=(10, 2))
ax.barh([0], [win_prob_100], color='blue', label='Team 100')
ax.barh([0], [win_prob_200], left=[win_prob_100], color='red', label='Team 200')
ax.set_xlim(0, 1)
ax.set_ylim(-0.5, 0.5)
ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_xticklabels(['0%', '25%', '50%', '75%', '100%'])
ax.set_yticks([])
ax.axvline(0.5, color='black', linestyle='--', linewidth=2, alpha=0.5)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=2)
ax.set_title('Win Probability', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='6-conclusion'></a>
# 6. Conclusion & Perspectives

## 6.1 Résultats Obtenus

### Performance du Modèle

✅ **61.4% accuracy** sur le test set (Gradient Boosting)
- Baseline (random) : 51.7%
- Amélioration : **+9.7 points**
- F1-score : 0.61-0.64 selon la classe

### Dataset

✅ **238 209 matchs** high-elo collectés
- Région : Korea (KR)
- Elo : Diamond I à Challenger
- Qualité : Données nettoyées et validées

### Features Engineering

✅ **3485 features** créées
- One-Hot encoding : 3366 features
- Playstyle (CommunityDragon) : 30 features
- OP.GG (matchups, synergies) : 42 features
- Composition d'équipe : 47 features

### Pipeline ML

✅ **Pipeline complet et automatisé**
- Collection via API Riot
- Preprocessing et feature engineering
- Training avec cross-validation
- Évaluation sur test set

---

## 6.2 Insights Clés

### 1. Matchups = Facteur #1

La feature `avg_matchup_advantage` représente **50.9%** de l'importance totale du modèle.

💡 **Implication** : Les avantages/désavantages en lane sont le facteur le plus prédictif de victoire.

### 2. Synergies Secondaires mais Importantes

Les features de synergies représentent **~4-5%** chacune de l'importance.

💡 **Implication** : Avoir des combos (ex: Yasuo + Knockup) améliore les chances mais n'est pas décisif.

### 3. Playstyle Features Contributives

Les scores de CC, damage, mobility contribuent à la prédiction.

💡 **Implication** : La composition d'équipe (tank, damage, utility) influence l'issue.

### 4. Meta Champions

Les champions les plus bannis sont souvent "broken" du patch.

💡 **Implication** : Le meta évolue rapidement, le modèle doit être réentraîné régulièrement.

---

## 6.3 Limitations

### 1. Accuracy Modeste (61.4%)

❌ Objectif initial : 70%+

**Raisons** :
- Facteur humain non pris en compte (skill des joueurs)
- Variance intrinsèque du jeu (RNG, décisions macro)
- Features bruitées (3485 features créent du surapprentissage)

### 2. One-Hot Encoding Limitatif

❌ N'encode pas les relations entre champions

**Solution** : Utiliser des embeddings (représentation dense)

### 3. Dataset Déséquilibré

❌ Certains champions rarement joués (< 10 apparitions)

**Impact** : Features One-Hot peu fiables pour ces champions

### 4. Meta Évolutive

❌ Patches toutes les 2 semaines changent l'équilibre

**Impact** : Modèle nécessite réentraînement régulier

---

## 6.4 Améliorations Futures

### 1. Algorithmes Avancés

🚀 **XGBoost / LightGBM**
- Meilleure gestion des features nombreuses
- Regularization intégrée
- Gain estimé : **+2-3%**

🚀 **Neural Networks**
- Embeddings pour champions
- Architecture : Input → Embedding → Dense Layers → Output
- Gain estimé : **+3-5%**

### 2. Feature Selection

🎯 Réduire de **3485 → 500-1000 features**
- Méthodes : Lasso, RFE, SHAP values
- Bénéfice : Réduction du bruit, meilleure généralisation

### 3. Embeddings de Champions

🧠 Remplacer One-Hot par **dense embeddings** (dimension 32-64)
- Capture les similarités entre champions
- Exemple : Garen et Darius auront des embeddings proches

### 4. Données Supplémentaires

📦 Collecter **500k+ matchs** de plusieurs régions
- EUW, NA, CN en plus de KR
- Meilleure généralisation

### 5. Features Temporelles

📅 Tracker **l'évolution du meta par patch**
- Win rate par champion et patch
- Détection de champions "broken" automatiquement

### 6. Intégration de Données de Gameplay

🎮 Ajouter des features **post-draft mais pre-game** :
- Elo moyen des joueurs
- Winrate historique des joueurs sur le champion sélectionné
- Nombre de parties jouées avec ce champion

---

## 6.5 Conclusion Finale

Ce projet démontre qu'il est **possible de prédire l'issue d'un match LoL à partir du draft uniquement**, avec une accuracy de **61.4%** (vs 51.7% baseline).

### Points Forts

✅ Pipeline end-to-end automatisé (collection → training → prédiction)

✅ Feature engineering riche (3485 features)

✅ Intégration de multiples sources de données (Riot API, OP.GG, CommunityDragon)

✅ Insights actionnables (matchups > synergies, importance de la composition)

### Applications Potentielles

🎯 **Coaching esport** : Aide à la décision en draft

🎯 **Outils de prédiction** : Site web / API pour joueurs

🎯 **Analyse de meta** : Détection de champions/combos forts

🎯 **Génération de tier lists** : Automatisation basée sur les données

---

### Merci !

---